<a href="https://colab.research.google.com/github/GAKUMAR3001/ML-and-NN/blob/main/Pipeline%20using%20PolynomialF_StandardS_LR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# 1. Create a dummy non-linear dataset
np.random.seed(42)
X = np.random.rand(100, 1) * 10
# y is a non-linear function with some random noise
y = 0.5 * (X**2) + 2 * X + 3 + np.random.randn(100, 1) * 3

# 2. Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create the Pipeline
# This chains Polynomial Features -> Standard Scaler -> Linear Regression
poly_model = make_pipeline(
    PolynomialFeatures(degree=2),
    StandardScaler(),
    LinearRegression()
)

# 4. Train the entire pipeline
# Under the hood: PolynomialFeatures fits & transforms X_train,
#                 StandardScaler fits & transforms that output,
#                 LinearRegression fits on the final scaled polynomial features.
poly_model.fit(X_train, y_train)

# 5. Predict using the pipeline
# Under the hood: It automatically transforms X_test through PolyFeatures and Scaler,
#                 then passes it to LinearRegression for prediction.
y_pred = poly_model.predict(X_test)

# 6. Evaluate
print(f"Test MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"Test R² Score: {r2_score(y_test, y_pred):.4f}")


# Accessing the learned coefficients of the Linear Regression step
coefficients = poly_model.named_steps['linearregression'].coef_
intercept = poly_model.named_steps['linearregression'].intercept_

print("Model Coefficients:", coefficients)
print("Model Intercept:", intercept)

Test MSE: 5.7226
Test R² Score: 0.9873
Model Coefficients: [[ 0.          3.40946789 16.83154579]]
Model Intercept: [27.42375627]



What is scikit-learn's make_pipeline?

In scikit-learn, make_pipeline is a utility function used to chain multiple data preprocessing steps and a machine learning model into a single, unified object.
Instead of manually scaling your training data, transforming your test data, and then passing it to the model step-by-step, a pipeline wraps everything together so it behaves like a single estimator.

Significance: Why is it so crucial?


1. Prevents Data Leakage (The Golden Rule)

Data leakage happens when information from outside the training dataset is used to train the model.
• Without a Pipeline: If you use scaler.fit_transform(X) on your entire dataset before splitting it into train/test, your training set "leaks" information about the mean and variance of your test set.
• With a Pipeline: When you call pipeline.fit(X_train, y_train), scikit-learn ensures that the transformers (like StandardScaler) only fit on the training data, and then simply transform the test data during pipeline.predict().

2. Automates and Simplifies Code

Instead of calling .fit() and .transform() separately on your training set and then remembering to only call .transform() on your test set, a pipeline handles the state transitions automatically. You only need to call .fit() and .predict().

To understand exactly how the .fit() function trains the entire pipeline under the hood, it helps to understand that scikit-learn divides its classes into two main categories:
1. Transformers: Steps that modify or process data (e.g., PolynomialFeatures, StandardScaler). They have .fit(), .transform(), and .fit_transform() methods.
2. Estimators/Models: The final step that makes predictions (e.g., LinearRegression). They only have .fit() and .predict() methods.
When you call poly_model.fit(X_train, y_train), scikit-learn executes a sequential chain reaction where the output of one step becomes the input for the next step.
Here is the exact step-by-step breakdown of what happens sequentially during that single line of code:

Step 1: PolynomialFeatures is Trained and Applied

The pipeline starts with your raw training data (X_train) and passes it into the first step.
• Fit: It looks at X_train to see how many features exist. It calculates how many new columns it needs to generate based on the degree=2 rule (e.g., if you give it \([x_1]\), it prepares to create \([1, x_1, x_1^2]\)).
• Transform: It actually creates those new columns.
• Output: It produces a new matrix, let's call it X_poly, which contains the original features plus their squared terms.

Step 2: StandardScaler is Trained and Applied

The pipeline takes the X_poly matrix from Step 1 and passes it into the StandardScaler. (Note: It does not look at your original raw X_train anymore).
• Fit: It calculates and remembers the mathematical mean (μ) and standard deviation (σ) of each column in the X_poly matrix.
• Transform: It subtracts the mean and divides by the standard deviation for every data point (\(z = \frac{x - \mu}{\sigma}\)).
• Output: It produces a newly scaled matrix, let's call it X_scaled.

Step 3: LinearRegression is Trained

Finally, the pipeline takes the X_scaled matrix from Step 2, along with your original target labels (y_train), and passes them into the final LinearRegression model.
• Fit: It runs the ordinary least squares gradient descent algorithm (or normal equation) using the X_scaled matrix and y_train. It calculates the optimal weights (w) and bias (b) for the model.
• Output: The training is complete. The pipeline objects now hold all the "learned" states (PolynomialFeatures knows the combinations, StandardScaler remembers the μ and σ, and LinearRegression holds the final optimal weights).

Visualizing the .fit() Chain Reaction

                [ X_train ]  (Raw Input)
                     |
                     v
   Step 1:  [ PolynomialFeatures ] --------> .fit_transform()
                     |
                     v
                [ X_poly ]   (Squared features matrix)
                     |
                     v
   Step 2:  [   StandardScaler   ] --------> .fit_transform() (Learns mean/std)
                     |
                     v
                [ X_scaled ] (Scaled & squared matrix)
                     |
                     v
   Step 3:  [  LinearRegression  ] + [ y_train ] -> .fit() (Finds optimal weights)

Why does this matter when you call .predict() later?

Because the pipeline sequentially executed .fit_transform() on the first two steps during training, it perfectly remembers the states.
When you later call poly_model.predict(X_test) on your completely unseen test data:
1. PolynomialFeatures only transforms X_test.
2. StandardScaler only transforms that output using the exact same mean and standard deviation it learned from the training set (It does not recalculate a new mean for the test set, preventing data leakage).
3. LinearRegression takes those final processed features and outputs your predictions (y_pred).
